# 05: MNIST MLP を学習し、SVD で圧縮する

## 一言で言うと
1. MNIST で MLP を学習する（ベースライン）
2. 学習済みの Linear 層を SVD で 2 層に置き換えて圧縮する
3. 精度・パラメータ数・MACs・推論時間を比較する

## 流れ

```text
学習（model）
  ↓
各 Linear を SVD → 2層化（two_layer_svd_model）
  ↓
Test Acc / params / MACs / 速度 を比較
```

## セルの役割

| あたり | 何をしているか |
|--------|----------------|
| import〜学習ループ | ベースライン MLP を学習 |
| SVD / devidetwolayer | 1層を低ランク2層に分解 |
| 評価・params・Acc差 | 圧縮の効果を数値で見る |
| MACs・benchmark | 計算量・実測速度の比較 |

## 用語
- **rank r**: 残す特異値の個数（小さいほど圧縮↑・精度↓しやすい）
- **compression_ratio**: 元params / 圧縮params（何倍小さくなったか）
- **MACs**: 乗算加算の回数（計算量の目安）

In [265]:
# ============================================================
# ライブラリの import
# ============================================================
import torch
import torch.nn as nn          # 層の定義（Linear, ReLU など）
import torch.optim as optim    # 最適化（Adam など）

from torch.utils.data import DataLoader          # ミニバッチでデータを流す
from torchvision import datasets, transforms     # MNIST と前処理

In [266]:
# ============================================================
# デバイス選択（GPU があれば cuda、なければ cpu）
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
# 今の環境は CPU でも MNIST MLP なら数分で学習できる

device: cuda


In [267]:
# ============================================================
# MNIST データの準備
# ============================================================
# ToTensor(): 画像を [0,1] の float テンソルに変換
#   元: PIL Image (28×28) → 後: Tensor (1, 28, 28)
transform = transforms.Compose([
    transforms.ToTensor()
])

# root="./data" にダウンロード＆キャッシュされる
# 初回だけダウンロード。LeCun サイトが 404 でもミラーから取れるので正常
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# DataLoader: データをミニバッチに分割して渡す
train_loader = DataLoader(
    train_dataset,
    batch_size=64,   # 1回の更新で使う枚数
    shuffle=True     # 学習時は順番を混ぜる
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1000,
    shuffle=False    # 評価時は混ぜなくてよい
)

print("train size:", len(train_dataset))  # 60000（規定値）
print("test size:", len(test_dataset))    # 10000（規定値）

train size: 60000
test size: 10000


In [268]:
# ============================================================
# 1バッチだけ取り出して shape を確認
# ============================================================
images, labels = next(iter(train_loader))

print("images shape:", images.shape)  # (64, 1, 28, 28) = バッチ, チャネル, H, W
print("labels shape:", labels.shape)  # (64,)
print("labels:", labels[:10])         # 0〜9 のクラスラベル

images shape: torch.Size([64, 1, 28, 28])
labels shape: torch.Size([64])
labels: tensor([4, 0, 9, 8, 9, 7, 0, 6, 8, 4])


In [269]:
# ============================================================
# MLP の定義
# ============================================================
# ポイント:
#   - forward の仕事は「画像 → 10クラスのスコア」まで通すこと
#   - SVD は forward の途中で毎回やらなくてよい（あとで重みに対して1回やる）
class MNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()

        # まずは普通の MLP（04と同じ構造）で学習する
        self.fc1 = nn.Linear(784, 512,bias=True)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(512, 256,bias=True)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 10,bias=True)

    def forward(self, x):
        # x: (batch, 1, 28, 28) → (batch, 784)
        x = x.view(x.size(0), -1)
        x = self.relu1(self.fc1(x))  # (batch, 512)
        x = self.relu2(self.fc2(x))  # (batch, 256)
        x = self.fc3(x)              # (batch, 10) ← CrossEntropy に渡す形

        return x

In [270]:
# ============================================================
# モデル・損失関数・最適化手法の準備
# ============================================================

# MNISTMLP() で、さっき定義したニューラルネット本体を作っています。
# .to(device) は、そのモデルをGPU に移しています。
model = MNISTMLP().to(device)

# 多クラス分類なので CrossEntropyLoss
# （内部で log_softmax + NLLLoss をやるので、モデル出力は logits のまま）
criterion = nn.CrossEntropyLoss()

# Adam: 学習率 0.001 はよく使う初期値
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

MNISTMLP(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=256, out_features=10, bias=True)
)


In [271]:
# ============================================================
# 学習 1 epoch = 「全データを1回見て、間違いを直す」
# ============================================================
# 1バッチごとのやること（これが学習の本体）:
#   1. 画像を入れる → 予測が出る
#   2. 正解と比べて loss（間違い）を計算
#   3. 「どの重みをどう直せばいいか」を計算（backward）
#   4. 重みを少し更新（step）
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    # model.train() は nn.Module が持っている仕組み
    # モデル自身とその子モジュールの training 状態を True にします
    # 今はなくてもあっても変わらない
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:  # 64枚ずつ取り出す
        images = images.to(device)
        labels = labels.to(device)

        # PyTorch は勾配を足し込むので、毎回リセット
        optimizer.zero_grad()            # ① 前回の修正メモを消す

        # forwardを使って計算している
        # outputs = model.forward(images)と同じ
        outputs = model(images)          # ② 予測（10クラス分のスコア）

        loss = criterion(outputs, labels)  # ③ 間違いの大きさ

        # 誤差逆伝播をして勾配を作る
        # forwardで計算したlossをbackwardで勾配を計算する
        loss.backward()                  # ④ どこを直せばいいか計算

        # backwardで計算した勾配をstepで更新する
        # 各層のパラメータを更新
        optimizer.step()                 # ⑤ 重みを少し直す

        # loss.item() はそのバッチの平均損失
        # images.size(0) を掛けるのは、64枚分の損失の合計っぽいものに直して、
        # あとで全体平均を取るため
        total_loss += loss.item() * images.size(0)

        # 0~9での各スコアを比較して１番大きいものをとる→これが予測
        predicted = torch.argmax(outputs, dim=1)  # 一番高いスコアの数字

        # 予測と正解を比較して、正解している数を数える
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total   # 平均 loss
    accuracy = correct / total      # 正解率

    return avg_loss, accuracy

In [272]:
# ============================================================
# テストデータでの評価（学習はしない）
# ============================================================
def evaluate(model, test_loader, criterion, device):
    model.eval()  # 評価モード

    total_loss = 0.0
    correct = 0
    total = 0

    # 評価時は勾配不要 → メモリ節約・高速化
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

In [273]:
# ============================================================
# 学習ループ（5 epoch）
# ============================================================
# 目安: Test Acc が 97〜98% くらいまで上がれば OK
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device)

    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, "
        f"Train Acc: {train_acc:.4f}, "
        f"Test Loss: {test_loss:.4f}, "
        f"Test Acc: {test_acc:.4f}"
    )

Epoch [1/5] Train Loss: 0.2370, Train Acc: 0.9300, Test Loss: 0.1015, Test Acc: 0.9691
Epoch [2/5] Train Loss: 0.0856, Train Acc: 0.9731, Test Loss: 0.1001, Test Acc: 0.9706
Epoch [3/5] Train Loss: 0.0590, Train Acc: 0.9815, Test Loss: 0.0843, Test Acc: 0.9734
Epoch [4/5] Train Loss: 0.0407, Train Acc: 0.9872, Test Loss: 0.0733, Test Acc: 0.9784
Epoch [5/5] Train Loss: 0.0315, Train Acc: 0.9896, Test Loss: 0.0649, Test Acc: 0.9808


In [274]:
# ============================================================
# SVD（学習の外で、重み行列 W を分解する）
# ============================================================
# W ≈ U_r @ diag(S_r) @ Vh_r
#   U_r : 左特異ベクトル（out × r）
#   S_r : 特異値（r,）← 対角行列の対角成分だけ
#   Vh_r: 右特異ベクトル（r × in）
def SVD(W, r):
    """重み行列 W を rank r で打ち切った SVD を返す"""
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)

    U_r = U[:, :r]   # 上位 r 列だけ残す
    S_r = S[:r]      # 上位 r 個の特異値
    Vh_r = Vh[:r, :] # 上位 r 行だけ残す

    return U_r, S_r, Vh_r

In [275]:
# ============================================================
# SVDで分解したものを再構築する
# ============================================================
def RebuildSVD(U_r, S_r, Vh_r,layer):
    W_r = U_r @ torch.diag(S_r) @ Vh_r

    new_layer = nn.Linear(
    layer.in_features,
    layer.out_features,
    bias=True,)

    with torch.no_grad():
        new_layer.weight.copy_(W_r)
        new_layer.bias.copy_(layer.bias)
    
    return new_layer.to(layer.weight.device)

In [276]:
# ============================================================
# 1層を2層に分けて計算コストを削減する
# ============================================================
# 元: Linear(in → out)
# 後: Linear(in → r, bias=False) + Linear(r → out, bias=True)
#
# 対応:
#   first.weight  = Vh_r
#   second.weight = U_r @ diag(S_r)
#   second.bias   = 元の bias
def devidetwolayer(layer,r):
    # 中間次元と低ランク数はいっしょにすべき？
    # → ここでは中間次元 = rank r として分解している
    U_r, S_r, Vh_r = SVD(layer.weight.detach(),r)  # detach: 勾配グラフから外す
    first_layer = nn.Linear(layer.in_features, r, bias=False)  # 中間次元 r、bias は不要
    second_layer = nn.Linear(r, layer.out_features, bias=True)  # 最終出力、bias はここに置く
    with torch.no_grad():  # 重みの代入時は勾配を計算しない
        first_layer.weight.copy_(Vh_r)                  # (r, in)
        second_layer.weight.copy_(U_r @ torch.diag(S_r))  # (out, r)
        second_layer.bias.copy_ (layer.bias.data)      # 元の bias をそのまま
    # 新規 Linear は CPU 上で作られるので、元の層と同じ device へ移す
    # 元の layer.weight がいる device を見て、新しく作った first_layer と second_layer を
    # 同じ場所に移してから返す、
    return nn.Sequential(first_layer,second_layer).to(layer.weight.device)


In [277]:
# ============================================================
# 2層に分解しせず、SVDを適用したモデルで評価
# ============================================================
# one_layer_svd_mode の骨格だけ作り、各 fc を「学習済み model の圧縮版」で差し替える

one_layer_svd_mode = MNISTMLP().to(device)
r1=64  # fc1: 784→512
# *は関数の結果を分解して渡す
r2=64
r3=10
one_layer_svd_mode.fc1 = RebuildSVD(*SVD(model.fc1.weight.detach(),r1),model.fc1)
r2=64  # fc2: 512→256
one_layer_svd_mode.fc2 = RebuildSVD(*SVD(model.fc2.weight.detach(),r2),model.fc2)
r3=10  # fc3: 256→10（もともと小さい層）
#one_layer_svd_mode.fc3 = RebuildSVD(*SVD(model.fc3.weight.detach(),r3),model.fc3)
one_layer_svd_mode.fc3 =model.fc3

# 圧縮後モデルでテスト精度を測る
time_comparison_test_loss, time_comparison_test_acc = evaluate(
        one_layer_svd_mode, test_loader, criterion, device)

# 注意: 下の print は学習直後の test_loss / test_acc（圧縮前）を表示している
print(
        f"Test Loss: {time_comparison_test_loss:.4f}, "
        f"Test Acc: {time_comparison_test_acc:.4f}"
    )

Test Loss: 0.0661, Test Acc: 0.9808


In [278]:
# ============================================================
# 2層に分解し、SVDを適用したモデルで評価
# ============================================================
# two_layer_svd_model の骨格だけ作り、各 fc を「学習済み model の圧縮版」で差し替える

two_layer_svd_model = MNISTMLP().to(device)

# modelに近似したものを入れる（rank が小さいほど圧縮↑）
r1=64  # fc1: 784→512
two_layer_svd_model.fc1 = devidetwolayer(model.fc1,r1)
r2=64  # fc2: 512→256
two_layer_svd_model.fc2 = devidetwolayer(model.fc2,r2)
r3=10  # fc3: 256→10（もともと小さい層）
#two_layer_svd_model.fc3 = devidetwolayer(model.fc3,r3)
two_layer_svd_model.fc3 = model.fc3

# 圧縮後モデルでテスト精度を測る
new_test_loss, new_test_acc = evaluate(
        two_layer_svd_model, test_loader, criterion, device)

# 注意: 下の print は学習直後の test_loss / test_acc（圧縮前）を表示している
print(
        f"Test Loss: {new_test_loss:.4f}, "
        f"Test Acc: {new_test_acc:.4f}"
    )

Test Loss: 0.0661, Test Acc: 0.9808


In [279]:
# ============================================================
# パラメータ数を数える（圧縮後と比較する基準）
# ============================================================
# fc1: 784*512+512 = 401,920
# fc2: 512*256+256 = 131,328
# fc3: 256*10+10   =   2,570
# 合計             = 535,818
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

baseline_params = count_parameters(model)   # 圧縮前
new_params=count_parameters(two_layer_svd_model)      # 圧縮後
print("Baseline params:", baseline_params)
print("New params:", new_params)

# compression_ratio = 元 / 圧縮後 → 大きいほど圧縮できている
compression_ratio = baseline_params / new_params
# 何割減ったか（例: 0.75 = 75%削減）
parameter_reduction_rate = 1.0 - new_params / baseline_params
print(f"Compression ratio: {compression_ratio:.2f}倍")
print(f"Parameter reduction: {parameter_reduction_rate:.2%}")


Baseline params: 535818
New params: 135434
Compression ratio: 3.96倍
Parameter reduction: 74.72%


In [280]:
# ============================================================
# 精度の低下量（圧縮のトレードオフ）
# ============================================================
# test_acc     : 圧縮前（学習直後）
# new_test_acc : 圧縮後
accuracy_drop = test_acc - new_test_acc

print("Baseline accuracy:", test_acc)
print("Compressed accuracy:", new_test_acc)
# accuracy_drop は float なので、Python 標準の round を使う
# （torch.round は Tensor 用）
print("Accuracy drop:", round(accuracy_drop, 5))

Baseline accuracy: 0.9808
Compressed accuracy: 0.9808
Accuracy drop: 0.0


In [281]:
# ============================================================
# MACs(Multiply-Accumulate Operations（乗算加算演算)
# ============================================================
# 1サンプルあたりの計算量の理論値
# Linear(in→out) ≈ in * out
# 圧縮後2層 ≈ in*r + r*out
def linear_macs(layer):
    #Linear層の1サンプル当たりのMACs
    return layer.in_features * layer.out_features

def compressed_linear_macs(in_features, out_features, rank):
    #SVD後の2層LinearのMACs
    return in_features * rank + rank * out_features

baseline_macs = (
    linear_macs(model.fc1)
    + linear_macs(model.fc2)
    + linear_macs(model.fc3)
)

compressed_macs = (
    compressed_linear_macs(784, 512, r1)
    + compressed_linear_macs(512, 256, r2)
    + compressed_linear_macs(256, 10, r3)
)

# 計算量が何割減ったか
compute_reduction_rate = 1.0 - compressed_macs / baseline_macs

print("Baseline MACs:", baseline_macs)
print("Compressed MACs:", compressed_macs)
print(f"Compute reduction: {compute_reduction_rate:.2%}")


Baseline MACs: 535040
Compressed MACs: 134756
Compute reduction: 74.81%


In [282]:
# ============================================================
# 実測の推論速度比較（1バッチあたりの平均時間）
# ============================================================
import time

def benchmark_inference(model, data_loader, device, warmup=10, repeats=5000):
    """1バッチ当たりの平均推論時間を測定する。"""
    # このコードでは意味ない
    # → 小さな MLP + GPU だと差が出にくい、という意味合い
    model.eval()

    images, _ = next(iter(data_loader))
    images = images.to(device)

    # ウォームアップ（初回の遅さを測りに入れない）
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(images)

    # GPU は非同期なので、計時前後で同期して正確に測る
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        for _ in range(repeats):
            _ = model(images)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    return elapsed / repeats  # 1回あたりの秒数


baseline_time = benchmark_inference(model, test_loader, device)
compressed_time = benchmark_inference(two_layer_svd_model, test_loader, device)
one_layer_svd_mode_time=benchmark_inference(one_layer_svd_mode, test_loader, device)

print(f"Baseline: {baseline_time * 1000:.3f} ms/batch")
print(f"TwolayerSVD: {compressed_time * 1000:.3f} ms/batch")
print(f"OneLayerSVD: {one_layer_svd_mode_time * 1000:.3f} ms/batch")
print(f"Speedup(baseline_time / TwolayerSVD): {baseline_time / compressed_time:.2f}x")  # >1 なら圧縮後の方が速い
print(f"Speedup(baseline_time / OneLayerSVD): {baseline_time / one_layer_svd_mode_time:.2f}x")  # >1 なら圧縮後の方が速い

Baseline: 0.126 ms/batch
TwolayerSVD: 0.144 ms/batch
OneLayerSVD: 0.125 ms/batch
Speedup(baseline_time / TwolayerSVD): 0.87x
Speedup(baseline_time / OneLayerSVD): 1.01x
